# PTM Binder Workshop: Designing Around `PVPNPD(PTR)EPIRKGQ`

In this notebook we'll build a phosphotyrosine peptide target, look at its bond graph, complete an RFD3 conditioning exercise, generate a binder backbone with RFD3, design the binder sequence with LigandMPNN, and then refold the result with RF3.

You can run it two ways:
1. Local Jupyter launched from the `ptm_foundry` repo.
2. Google Colab with a GPU runtime.

Here's the flow:
1. Get the repo and runtime set up.
2. Spoof a CIF for the PTM peptide target on the fly.
3. Check the target bond graph so we know the phosphotyrosine chemistry is really there.
4. Complete the RFD3 JSON exercise for a `100` residue binder against the phosphorylated peptide.
5. Generate a binder backbone with RFD3 and design the binder sequence with LigandMPNN.
6. Refold the selected design with RF3 and save the structures we care about.


## 0. Setup

This notebook works in both Google Colab and a local Pixi setup.

- In Colab: switch to a GPU runtime and run the setup cells from the top.
- Locally: launch Jupyter from the `ptm_foundry` repo in the Pixi `dev` environment.

Recommended local setup from the repo root:

```bash
pixi install -e dev
pixi run -e dev install-workshop-kernel
pixi run -e dev workshop-notebook
```

Then open this notebook with the `PTM Workshop` kernel and run it from the top.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
GIT_URL = os.environ.get("PTM_FOUNDRY_GIT_URL", "https://github.com/magnusbauer/ptm_foundry.git")
GIT_REF = os.environ.get("PTM_FOUNDRY_GIT_REF", "production")

if IN_COLAB:
    REPO_DIR = Path("/content/foundry")
else:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent]
    REPO_DIR = next(
        (candidate for candidate in candidates if (candidate / "examples").exists() and (candidate / "models").exists()),
        cwd,
    )

if IN_COLAB and not REPO_DIR.exists():
    subprocess.check_call(
        ["git", "clone", "--branch", GIT_REF, GIT_URL, str(REPO_DIR)]
    )

SOURCE_PATHS = [
    REPO_DIR / "src",
    REPO_DIR / "models" / "rfd3" / "src",
    REPO_DIR / "models" / "mpnn" / "src",
    REPO_DIR / "models" / "rf3" / "src",
    REPO_DIR / "examples",
]
for source_path in SOURCE_PATHS:
    if source_path.exists() and str(source_path) not in sys.path:
        sys.path.insert(0, str(source_path))

print(f"Running in Colab: {IN_COLAB}")
print(f"Repository root: {REPO_DIR}")
if IN_COLAB:
    print(f"Clone source:    {GIT_URL} @ {GIT_REF}")


In [ ]:
SPOOF_CIF_PATH = REPO_DIR / "examples" / "spoof_cif.py"
WORKSHOP_OUTPUT_DIR = REPO_DIR / "examples" / "workshop_outputs"

if not SPOOF_CIF_PATH.exists():
    raise FileNotFoundError(
        f"Expected workshop helper at {SPOOF_CIF_PATH}. "
        "Make sure the repo checkout includes examples/spoof_cif.py."
    )

WORKSHOP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Using workshop helper: {SPOOF_CIF_PATH}")
print(f"Workshop outputs:     {WORKSHOP_OUTPUT_DIR}")


In [ ]:
%%time

import importlib.util
import os
import subprocess
import sys
from pathlib import Path
from urllib.request import urlretrieve

os.environ["CCD_MIRROR_PATH"] = ""
os.environ["PDB_MIRROR_PATH"] = ""

for env_name, default_value in {
    "DEBUG": "0",
    "TYPE_CHECK": "0",
    "NAN_CHECK": "1",
    "DISABLE_CUEQUIVARIANCE": "0",
}.items():
    if not os.environ.get(env_name, "").strip():
        os.environ[env_name] = default_value

CKPT_DIR = Path.home() / ".foundry" / "checkpoints"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
os.environ["FOUNDRY_CHECKPOINTS_DIR"] = str(CKPT_DIR)


def ensure_pip() -> None:
    if importlib.util.find_spec("pip") is None:
        subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])


def pip_install(packages: list[str]) -> None:
    if not packages:
        return
    ensure_pip()
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])


def download_file(url: str, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp_dest = dest.with_suffix(dest.suffix + ".part")
    urlretrieve(url, tmp_dest)
    tmp_dest.replace(dest)


if IN_COLAB:
    subprocess.run(
        [sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"],
        check=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

REQUIRED_PACKAGES = {
    "matplotlib": "matplotlib",
    "networkx": "networkx",
    "pandas": "pandas",
    "hydride": "hydride",
    "biotite": "biotite",
    "atomworks": "atomworks[ml]>=2.1.1",
    "lightning": "lightning>=2.5.0",
    "rootutils": "rootutils>=1.0.7,<1.1",
    "hydra": "hydra-core>=1.3.0,<1.4",
    "environs": "environs>=11.0.0,<12",
    "rich": "rich>=13.9.4",
    "jaxtyping": "jaxtyping>=0.2.17,<1",
    "beartype": "beartype>=0.18.0,<1",
    "loralib": "loralib>=0.1.1",
    "einops": "einops>=0.8.0,<1",
    "einx": "einx>=0.1.0,<1",
    "opt_einsum": "opt_einsum>=3.4.0,<4",
    "tree": "dm-tree>=0.1.6,<1",
    "zstandard": "zstandard",
    "toolz": "toolz",
    "pydantic": "pydantic>=2.8",
}
missing_packages = [
    package
    for module_name, package in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]
if missing_packages:
    print("Installing missing packages:")
    for package in missing_packages:
        print(" -", package)
    pip_install(missing_packages)
else:
    print("Python dependencies already satisfied.")

CHECKPOINTS = {
    "rfd3": {
        "url": "https://files.ipd.uw.edu/pub/rfd3/rfd3_foundry_2025_12_01_remapped.ckpt",
        "filename": "rfd3_latest.ckpt",
    },
    "ligandmpnn": {
        "url": "https://files.ipd.uw.edu/pub/ligandmpnn/ligandmpnn_v_32_010_25.pt",
        "filename": "ligandmpnn_v_32_010_25.pt",
    },
    "rf3": {
        "url": "https://files.ipd.uw.edu/pub/rf3/rf3_foundry_01_24_latest_remapped.ckpt",
        "filename": "rf3_foundry_01_24_latest_remapped.ckpt",
    },
}

for name, info in CHECKPOINTS.items():
    dest = CKPT_DIR / info["filename"]
    if dest.exists():
        print(f"{name}: already present at {dest}")
        continue
    print(f"Downloading {name} -> {dest}")
    download_file(info["url"], dest)

print("\nCheckpoint directory contents:")
for checkpoint_path in sorted(CKPT_DIR.iterdir()):
    print(" -", checkpoint_path.name)

RFD3_CKPT = CKPT_DIR / CHECKPOINTS["rfd3"]["filename"]
LIGANDMPNN_CKPT = CKPT_DIR / CHECKPOINTS["ligandmpnn"]["filename"]
RF3_CKPT = CKPT_DIR / CHECKPOINTS["rf3"]["filename"]

print("\nResolved checkpoint paths:")
print(f" - RFD3:       {RFD3_CKPT}")
print(f" - LigandMPNN: {LIGANDMPNN_CKPT}")
print(f" - RF3:        {RF3_CKPT}")


In [ ]:
import json
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore", module="atomworks")

EXAMPLES_DIR = REPO_DIR / "examples"
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from atomworks.constants import PROTEIN_BACKBONE_ATOM_NAMES
from atomworks.io.utils.io_utils import to_cif_file
from atomworks.io.utils.visualize import view
from biotite.structure import rmsd, superimpose
from lightning.fabric import seed_everything
from spoof_cif import (
    bond_table_for_residue,
    build_rfd3_input_template,
    chain_summary,
    extract_chain_sequence,
    find_ptm_positions,
    plot_atom_bond_graph_with_types,
    plot_local_atom_bond_graph,
    plot_residue_bond_graph,
    ptr_selector_guidance,
    spoof_cif_from_sequence,
    tokenize_polymer_sequence,
)


## 1. Define the Workshop Target

The peptide target is `PVPNPD(PTR)EPIRKGQ`, where `(PTR)` is phosphotyrosine. We are going to build a binder around that site, so first let's make sure the residue numbering and PTM position line up the way we expect.


In [ ]:
TARGET_SEQUENCE = "PVPNPD(PTR)EPIRKGQ"
TARGET_CHAIN_ID = "B"
BINDER_LENGTH = 100
EXAMPLE_NAME = "ptr_workshop"
WORK_DIR = Path("/content/ptm_workshop") if IN_COLAB else WORKSHOP_OUTPUT_DIR
WORK_DIR.mkdir(parents=True, exist_ok=True)

print(f"Target sequence: {TARGET_SEQUENCE}")
print(f"Target chain:    {TARGET_CHAIN_ID}")
print(f"Binder length:   {BINDER_LENGTH}")
print(f"Working dir:     {WORK_DIR}")


In [ ]:
tokens = tokenize_polymer_sequence(TARGET_SEQUENCE)
ptm_positions = find_ptm_positions(tokens, {"PTR"})

display(
    pd.DataFrame(
        {
            "residue_id": np.arange(1, len(tokens) + 1),
            "token": tokens,
            "is_ptr": [token == "PTR" for token in tokens],
        }
    )
)
print(f"PTM positions: {ptm_positions}")


## 2. Spoof the PTM CIF on the Fly

This workshop does not rely on a committed `ptr_workshop.cif`. Instead, we regenerate the phosphopeptide target directly from the sequence each time so the PTM chemistry is explicit and reproducible.

The peptide is fixed in sequence because chain `B` comes from this input CIF and later `LigandMPNN` only designs chain `A`. The peptide can still change in structure because `select_fixed_atoms` stays `false`, so RFD3 is allowed to move the coordinates even while the residue identities and PTM chemistry stay fixed.


In [ ]:
cif_path, spoofed_target = spoof_cif_from_sequence(
    name=EXAMPLE_NAME,
    sequence=TARGET_SEQUENCE,
    out_dir=WORK_DIR,
    chain_id=TARGET_CHAIN_ID,
)
ptm_residue_id = next(iter(ptm_positions))
selector_guidance = ptr_selector_guidance(TARGET_CHAIN_ID, ptm_residue_id)
rfd3_template = build_rfd3_input_template(
    name=EXAMPLE_NAME,
    cif_path=cif_path,
    binder_length=BINDER_LENGTH,
    target_length=len(tokens),
    target_chain_id=TARGET_CHAIN_ID,
)
RFD3_JSON_PATH = WORK_DIR / f"{EXAMPLE_NAME}.json"

print(f"Spoofed CIF:   {cif_path}")
print(f"PTM residue:   {TARGET_CHAIN_ID}{ptm_residue_id}")
print(f"Sequence len:  {len(tokens)}")
print(f"JSON path:     {RFD3_JSON_PATH}")
print(json.dumps(selector_guidance, indent=2))


In [ ]:
view(spoofed_target)


## 3. Inspect the Bond Graph

Before running design, it helps to look at the chemistry from a few angles. The next cells show:
- a bond table centered on `PTR`
- a residue-level bond graph for the full target chain
- an atom-by-atom local graph around the PTM and its peptide neighbors
- a larger 2D atom graph with the bond types written on each edge


In [ ]:
bond_table = bond_table_for_residue(
    spoofed_target,
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
    include_neighbors=True,
)
display(bond_table)


In [ ]:
plot_residue_bond_graph(spoofed_target, chain_id=TARGET_CHAIN_ID)
plt.show()


In [ ]:
plot_local_atom_bond_graph(
    spoofed_target,
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(13, 9))
plot_local_atom_bond_graph(
    spoofed_target,
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
    ax=ax,
)
plt.tight_layout()
plt.show()


## 4. Exercise: Write the RFD3 JSON

Your assignment is to finish the selector fields for a `100` residue binder against the phosphorylated peptide.

Use the bond tables and graphs above to decide what goes into:
- `select_hotspots`: atoms that should anchor interface orientation around the PTM.
- `select_buried`: atoms you want packed against the binder.
- `select_hbond_acceptor`: atoms that should accept H-bonds from the binder.

Keep the rest of the JSON scaffold unchanged.

About `OH`: in `PTR`, `OH` is the tyrosine side-chain oxygen that links the aromatic ring to the phosphate. It is part of the phosphotyrosine residue, not a separate free hydroxyl group floating off the PTM.


In [ ]:
rfd3_input = json.loads(json.dumps(rfd3_template))

# Exercise: fill these selector maps in before running the next cell.
rfd3_input[EXAMPLE_NAME]["select_hotspots"] = {
}
rfd3_input[EXAMPLE_NAME]["select_buried"] = {
}
rfd3_input[EXAMPLE_NAME]["select_hbond_acceptor"] = {
}

print(json.dumps(rfd3_input, indent=2))
print("\nSelector guidance:")
print(json.dumps(selector_guidance, indent=2))


In [ ]:
missing_fields = [
    field
    for field in ("select_hotspots", "select_buried", "select_hbond_acceptor")
    if not rfd3_input[EXAMPLE_NAME][field]
]
if missing_fields:
    raise ValueError(
        "Fill in the selector exercise before running RFD3. Missing: "
        + ", ".join(missing_fields)
    )

RFD3_JSON_PATH.write_text(json.dumps(rfd3_input, indent=2))
print(f"Wrote RFD3 JSON to {RFD3_JSON_PATH}")
print(json.dumps(rfd3_input, indent=2))


## 5. Generate a Binder Backbone with RFD3

Now we can run RFD3 on the spoofed PTM peptide. To keep the workshop moving, this example makes a single binder backbone (`diffusion_batch_size=1`, `n_batches=1`).


In [ ]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
print(f"CUDA device count: {torch.cuda.device_count()}")
if cuda_available:
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("No CUDA device detected. The RFD3 / LigandMPNN / RF3 cells are workshop GPU steps and can be extremely slow on CPU.")


In [ ]:
from rfd3.engine import RFD3InferenceConfig, RFD3InferenceEngine

seed_everything(7)

rfd3_config = RFD3InferenceConfig(
    ckpt_path=str(RFD3_CKPT),
    diffusion_batch_size=1,
)
rfd3_engine = RFD3InferenceEngine(**rfd3_config)
rfd3_outputs = rfd3_engine.run(
    inputs=str(RFD3_JSON_PATH),
    out_dir=None,
    n_batches=1,
)

print(f"Output keys: {list(rfd3_outputs.keys())}")


In [ ]:
first_key = next(iter(rfd3_outputs))
rfd3_complex = rfd3_outputs[first_key][0].atom_array

display(chain_summary(rfd3_complex))
view(rfd3_complex)


## 6. Design Only the Binder Chain with LigandMPNN

RFD3 gives us a binder backbone plus the fixed peptide target. Next we use LigandMPNN to assign amino acids to the binder chain while leaving the PTM peptide unchanged. The important setting is `designed_chains=["A"]`.


In [ ]:
from mpnn.inference_engines.mpnn import MPNNInferenceEngine

mpnn_engine = MPNNInferenceEngine(
    model_type="ligand_mpnn",
    checkpoint_path=str(LIGANDMPNN_CKPT),
    is_legacy_weights=True,
    out_directory=None,
    write_structures=False,
    write_fasta=False,
)
mpnn_outputs = mpnn_engine.run(
    input_dicts=[
        {
            "batch_size": 4,
            "remove_waters": True,
            "designed_chains": ["A"],
        }
    ],
    atom_arrays=[rfd3_complex],
)

print(f"Generated {len(mpnn_outputs)} sequence proposals.")


In [ ]:
mpnn_summary = []
for output in mpnn_outputs:
    mpnn_summary.append(
        {
            "design_idx": output.output_dict["design_idx"],
            "binder_sequence": extract_chain_sequence(output.atom_array, "A"),
            "target_sequence": extract_chain_sequence(output.atom_array, TARGET_CHAIN_ID),
            "sequence_recovery": output.output_dict["sequence_recovery"],
            "ligand_interface_sequence_recovery": output.output_dict[
                "ligand_interface_sequence_recovery"
            ],
        }
    )

display(pd.DataFrame(mpnn_summary))


In [ ]:
selected_design = mpnn_outputs[0]
selected_complex = selected_design.atom_array

print("Selected binder chain A sequence:")
print(extract_chain_sequence(selected_complex, "A"))
print("\nTarget chain B sequence:")
print(extract_chain_sequence(selected_complex, TARGET_CHAIN_ID))
view(selected_complex)


## 7. Refold the Selected Complex with RF3

Now let's do a quick RF3 refold. The idea here is simple: take the designed sequence, refold it in the full complex, and see whether the binder still looks consistent with the backbone we just made. It's a fast sanity check, not a final ranking step.


In [ ]:
from rf3.inference_engines.rf3 import RF3InferenceEngine
from rf3.utils.inference import InferenceInput

rf3_engine = RF3InferenceEngine(ckpt_path=str(RF3_CKPT), verbose=False)
rf3_input = InferenceInput.from_atom_array(
    selected_complex,
    example_id=EXAMPLE_NAME,
)
rf3_outputs = rf3_engine.run(
    inputs=rf3_input,
    annotate_b_factor_with_plddt=True,
)
rf3_output = rf3_outputs[EXAMPLE_NAME][0]
summary = rf3_output.summary_confidences

print(json.dumps(summary, indent=2, default=float))


In [ ]:
view(rf3_output.atom_array)


In [ ]:
binder_generated = selected_complex[
    (selected_complex.chain_id == "A")
    & np.isin(selected_complex.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
]
binder_refolded = rf3_output.atom_array[
    (rf3_output.atom_array.chain_id == "A")
    & np.isin(rf3_output.atom_array.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
]

generated_lookup = {
    (binder_generated.chain_id[i], int(binder_generated.res_id[i]), binder_generated.atom_name[i]): i
    for i in range(len(binder_generated))
}
refolded_lookup = {
    (binder_refolded.chain_id[i], int(binder_refolded.res_id[i]), binder_refolded.atom_name[i]): i
    for i in range(len(binder_refolded))
}

common_keys = [key for key in generated_lookup if key in refolded_lookup]
if not common_keys:
    raise ValueError("No common binder backbone atoms found for the RF3 self-consistency check")

generated_idx = np.array([generated_lookup[key] for key in common_keys], dtype=int)
refolded_idx = np.array([refolded_lookup[key] for key in common_keys], dtype=int)

binder_generated_common = binder_generated[generated_idx]
binder_refolded_common = binder_refolded[refolded_idx]

binder_refolded_fitted, _ = superimpose(binder_generated_common, binder_refolded_common)
binder_rmsd = rmsd(binder_generated_common, binder_refolded_fitted)
print(f"Binder backbone self-consistency RMSD after RF3 refolding: {binder_rmsd:.2f} A")
print(f"Used {len(common_keys)} matched backbone atoms ({len(binder_generated)} generated vs {len(binder_refolded)} refolded before matching).")


## 8. Notebook Checks on the RF3 Complex

Now let's do the same checks right here in the notebook so you can see what each one is telling you. We will look at RMSD after binder-based alignment, phosphosite H-bonds, and how buried the phosphate group is.


In [ ]:
import biotite.structure as struc
import hydride
import py3Dmol
from atomworks.io.utils.io_utils import to_cif_string

HBOND_CUTOFF_DIST = 2.5
HBOND_CUTOFF_ANGLE = 120.0
HBOND_PH = 7.0
HBOND_REQUIRE_CROSS_RESIDUE = True
SASA_PROBE_RADIUS = 1.4
SASA_VDW_RADII = "Single"
SASA_POINT_NUMBER = 1000
SASA_POINT_DISTR = "Fibonacci"
FINAL_FILTER_MAX_PEPTIDE_CA_RMSD = 1.5
FINAL_FILTER_MIN_PO4_BURIAL = 0.35
FINAL_FILTER_MIN_HBONDS = 2


In [ ]:
reference_complex = selected_complex
mobile_complex = rf3_output.atom_array

PHOSPHATE_ATOMS = ("P", "O1P", "O2P", "O3P")

binder_backbone_mask_ref = (
    (reference_complex.chain_id == "A")
    & np.isin(reference_complex.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
)
binder_backbone_mask_mobile = (
    (mobile_complex.chain_id == "A")
    & np.isin(mobile_complex.atom_name, PROTEIN_BACKBONE_ATOM_NAMES)
)

peptide_mask_ref = reference_complex.chain_id == TARGET_CHAIN_ID
peptide_mask_mobile = mobile_complex.chain_id == TARGET_CHAIN_ID

peptide_ca_mask_ref = peptide_mask_ref & (reference_complex.atom_name == "CA")
peptide_ca_mask_mobile = peptide_mask_mobile & (mobile_complex.atom_name == "CA")

ptr_mask_ref = (
    peptide_mask_ref
    & (reference_complex.res_id == ptm_residue_id)
    & (reference_complex.res_name == "PTR")
)
ptr_mask_mobile = (
    peptide_mask_mobile
    & (mobile_complex.res_id == ptm_residue_id)
    & (mobile_complex.res_name == "PTR")
)

po4_mask_ref = ptr_mask_ref & np.isin(reference_complex.atom_name, PHOSPHATE_ATOMS)
po4_mask_mobile = ptr_mask_mobile & np.isin(mobile_complex.atom_name, PHOSPHATE_ATOMS)

selection_summary = pd.DataFrame(
    [
        {"selection": "binder_backbone", "reference_atoms": int(binder_backbone_mask_ref.sum()), "mobile_atoms": int(binder_backbone_mask_mobile.sum())},
        {"selection": "whole_peptide", "reference_atoms": int(peptide_mask_ref.sum()), "mobile_atoms": int(peptide_mask_mobile.sum())},
        {"selection": "peptide_ca", "reference_atoms": int(peptide_ca_mask_ref.sum()), "mobile_atoms": int(peptide_ca_mask_mobile.sum())},
        {"selection": "ptr_residue", "reference_atoms": int(ptr_mask_ref.sum()), "mobile_atoms": int(ptr_mask_mobile.sum())},
        {"selection": "po4_group", "reference_atoms": int(po4_mask_ref.sum()), "mobile_atoms": int(po4_mask_mobile.sum())},
    ]
)

display(selection_summary)


In [ ]:
def paired_common_indices(reference, mobile, ref_mask, mobile_mask):
    ref_indices = np.flatnonzero(ref_mask)
    mobile_indices = np.flatnonzero(mobile_mask)

    ref_lookup = {
        (reference.chain_id[i], int(reference.res_id[i]), reference.atom_name[i]): i
        for i in ref_indices
    }
    mobile_lookup = {
        (mobile.chain_id[i], int(mobile.res_id[i]), mobile.atom_name[i]): i
        for i in mobile_indices
    }

    common_keys = [key for key in ref_lookup if key in mobile_lookup]
    if not common_keys:
        raise ValueError("No common atoms found between the reference and mobile selections")

    ref_paired = np.array([ref_lookup[key] for key in common_keys], dtype=int)
    mobile_paired = np.array([mobile_lookup[key] for key in common_keys], dtype=int)
    return ref_paired, mobile_paired


def align_mobile_on_binder_backbone(reference, mobile, ref_mask, mobile_mask):
    ref_idx, mobile_idx = paired_common_indices(reference, mobile, ref_mask, mobile_mask)
    _, transform = struc.superimpose(reference[ref_idx], mobile[mobile_idx])
    mobile_aligned = transform.apply(mobile)
    alignment_rmsd = float(struc.rmsd(reference[ref_idx], mobile_aligned[mobile_idx]))
    return mobile_aligned, transform, alignment_rmsd


def rmsd_for_masks(reference, mobile, ref_mask, mobile_mask, allow_mismatch=False):
    if allow_mismatch:
        ref_idx, mobile_idx = paired_common_indices(reference, mobile, ref_mask, mobile_mask)
    else:
        ref_idx = np.flatnonzero(ref_mask)
        mobile_idx = np.flatnonzero(mobile_mask)
        if len(ref_idx) != len(mobile_idx):
            raise ValueError(
                f"Selection size mismatch: reference={len(ref_idx)} mobile={len(mobile_idx)}"
            )

    if len(ref_idx) == 0:
        raise ValueError("Selection matched no atoms")

    metric_rmsd = float(struc.rmsd(reference[ref_idx], mobile[mobile_idx]))
    return metric_rmsd, len(ref_idx)


def atom_triplet_label(atom_array, atom_index):
    return (
        f"{atom_array.chain_id[atom_index]}:"
        f"{atom_array.res_name[atom_index]}{atom_array.res_id[atom_index]}:"
        f"{atom_array.atom_name[atom_index]}"
    )


def same_residue(atom_array, atom_i, atom_j):
    return (
        atom_array.chain_id[atom_i] == atom_array.chain_id[atom_j]
        and atom_array.res_id[atom_i] == atom_array.res_id[atom_j]
    )


def format_hbond_connection(atom_array, donor_idx, hydrogen_idx, acceptor_idx):
    donor = atom_triplet_label(atom_array, donor_idx)
    hydrogen = atom_array.atom_name[hydrogen_idx]
    acceptor = atom_triplet_label(atom_array, acceptor_idx)
    return f"{donor} -- {hydrogen} --> {acceptor}"


def prepare_atom_array_for_hbonds(atom_array, ph=HBOND_PH):
    prepared = atom_array.copy()
    if "H" in prepared.element:
        prepared = prepared[prepared.element != "H"]

    for category in list(prepared.get_annotation_categories()):
        annotation = prepared.get_annotation(category)
        if getattr(annotation, "ndim", 1) != 1:
            prepared.del_annotation(category)

    prepared.bonds = struc.connect_via_residue_names(prepared)
    prepared.charge = hydride.estimate_amino_acid_charges(prepared, ph=ph)
    prepared_with_h, _ = hydride.add_hydrogen(prepared)
    prepared_with_h.coord = hydride.relax_hydrogen(prepared_with_h)
    return prepared_with_h


def compute_phosphosite_hbond_metrics(atom_array, chain_id, residue_id, res_name="PTR"):
    prepared = prepare_atom_array_for_hbonds(atom_array, ph=HBOND_PH)
    phosphosite_mask = (
        (prepared.chain_id == chain_id)
        & (prepared.res_id == residue_id)
        & (prepared.res_name == res_name)
    )
    if not phosphosite_mask.any():
        raise ValueError(f"No atoms found for {chain_id}:{res_name}{residue_id}")

    hbond_result = struc.hbond(
        prepared,
        selection1_type="both",
        cutoff_dist=HBOND_CUTOFF_DIST,
        cutoff_angle=HBOND_CUTOFF_ANGLE,
    )
    triplets = hbond_result[0] if isinstance(hbond_result, tuple) else hbond_result

    all_connections = []
    phosphosite_connections = []
    phosphosite_records = []
    for donor_idx, hydrogen_idx, acceptor_idx in triplets:
        if HBOND_REQUIRE_CROSS_RESIDUE and same_residue(prepared, donor_idx, acceptor_idx):
            continue

        connection = format_hbond_connection(
            prepared,
            donor_idx=donor_idx,
            hydrogen_idx=hydrogen_idx,
            acceptor_idx=acceptor_idx,
        )
        all_connections.append(connection)

        donor_match = bool(phosphosite_mask[donor_idx])
        acceptor_match = bool(phosphosite_mask[acceptor_idx])
        if donor_match or acceptor_match:
            phosphosite_connections.append(connection)
            phosphosite_records.append(
                {
                    "donor_idx": int(donor_idx),
                    "hydrogen_idx": int(hydrogen_idx),
                    "acceptor_idx": int(acceptor_idx),
                    "donor_label": atom_triplet_label(prepared, donor_idx),
                    "hydrogen_label": atom_triplet_label(prepared, hydrogen_idx),
                    "acceptor_label": atom_triplet_label(prepared, acceptor_idx),
                    "donor_acceptor_distance": float(
                        np.linalg.norm(prepared.coord[donor_idx] - prepared.coord[acceptor_idx])
                    ),
                    "hydrogen_acceptor_distance": float(
                        np.linalg.norm(prepared.coord[hydrogen_idx] - prepared.coord[acceptor_idx])
                    ),
                }
            )

    return {
        "prepared_structure": prepared,
        "total_hbonds": float(len(all_connections)),
        "phosphosite_hbonds": float(len(phosphosite_connections)),
        "connections": all_connections,
        "phosphosite_connections": phosphosite_connections,
        "phosphosite_records": phosphosite_records,
    }


def make_structure_overlay_view(reference, mobile, zoom_to_selection=None, width=700, height=500):
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(
        to_cif_string(
            reference,
            include_entity_poly=False,
            _allow_ambiguous_bond_annotations=True,
        ),
        "mmcif",
    )
    viewer.addModel(
        to_cif_string(
            mobile,
            include_entity_poly=False,
            _allow_ambiguous_bond_annotations=True,
        ),
        "mmcif",
    )
    viewer.setStyle(
        {"model": 0},
        {"cartoon": {"color": "#9ca3af", "opacity": 0.55}, "stick": {"colorscheme": "grayCarbon", "radius": 0.12}},
    )
    viewer.setStyle(
        {"model": 1},
        {"cartoon": {"color": "#2563eb", "opacity": 0.85}, "stick": {"colorscheme": "cyanCarbon", "radius": 0.12}},
    )
    if zoom_to_selection is not None:
        viewer.setStyle(
            {"model": 0, **zoom_to_selection},
            {"stick": {"colorscheme": "grayCarbon", "radius": 0.2}, "sphere": {"scale": 0.18}},
        )
        viewer.setStyle(
            {"model": 1, **zoom_to_selection},
            {"stick": {"colorscheme": "cyanCarbon", "radius": 0.2}, "sphere": {"scale": 0.18}},
        )
        viewer.zoomTo(zoom_to_selection)
    else:
        viewer.zoomTo()
    return viewer


def make_hbond_view(atom_array, hbond_records, chain_id, residue_id, width=700, height=500):
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(
        to_cif_string(
            atom_array,
            include_entity_poly=False,
            _allow_ambiguous_bond_annotations=True,
        ),
        "mmcif",
    )
    viewer.setStyle(
        {},
        {"cartoon": {"color": "#94a3b8", "opacity": 0.45}, "stick": {"colorscheme": "lightgrayCarbon", "radius": 0.1}},
    )
    viewer.setStyle(
        {"chain": chain_id, "resi": int(residue_id)},
        {"stick": {"colorscheme": "orangeCarbon", "radius": 0.18}, "sphere": {"scale": 0.22}},
    )
    for record in hbond_records:
        donor = atom_array.coord[record["donor_idx"]]
        acceptor = atom_array.coord[record["acceptor_idx"]]
        viewer.addSphere(
            {
                "center": {"x": float(donor[0]), "y": float(donor[1]), "z": float(donor[2])},
                "radius": 0.35,
                "color": "#2563eb",
                "opacity": 0.9,
            }
        )
        viewer.addSphere(
            {
                "center": {"x": float(acceptor[0]), "y": float(acceptor[1]), "z": float(acceptor[2])},
                "radius": 0.35,
                "color": "#f59e0b",
                "opacity": 0.9,
            }
        )
        viewer.addLine(
            {
                "start": {"x": float(donor[0]), "y": float(donor[1]), "z": float(donor[2])},
                "end": {"x": float(acceptor[0]), "y": float(acceptor[1]), "z": float(acceptor[2])},
                "color": "#f59e0b",
                "dashed": True,
            }
        )
    viewer.zoomTo({"chain": chain_id, "resi": int(residue_id)})
    return viewer


def project_points_to_plane(coords):
    centered = coords - coords.mean(axis=0, keepdims=True)
    _, _, vh = np.linalg.svd(centered, full_matrices=False)
    basis = vh[:2]
    return centered @ basis.T


def plot_hbond_contact_map(atom_array, hbond_records, chain_id, residue_id, phosphate_atom_names, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(8, 7))

    if not hbond_records:
        ax.text(0.5, 0.5, "No phosphosite H-bonds detected", ha="center", va="center", transform=ax.transAxes)
        ax.set_axis_off()
        return ax

    residue_mask = (atom_array.chain_id == chain_id) & (atom_array.res_id == residue_id)
    phosphate_mask = residue_mask & np.isin(atom_array.atom_name, phosphate_atom_names)

    residue_indices = set(np.flatnonzero(residue_mask))
    phosphate_indices = set(np.flatnonzero(phosphate_mask))
    donor_indices = {int(record["donor_idx"]) for record in hbond_records}
    acceptor_indices = {int(record["acceptor_idx"]) for record in hbond_records}
    selected_indices = np.array(sorted(residue_indices | donor_indices | acceptor_indices), dtype=int)

    projected = project_points_to_plane(atom_array.coord[selected_indices])
    point_by_index = {atom_idx: projected[pos] for pos, atom_idx in enumerate(selected_indices)}

    for record in hbond_records:
        donor_xy = point_by_index[int(record["donor_idx"])]
        acceptor_xy = point_by_index[int(record["acceptor_idx"])]
        ax.plot(
            [donor_xy[0], acceptor_xy[0]],
            [donor_xy[1], acceptor_xy[1]],
            linestyle="--",
            linewidth=1.8,
            color="#f59e0b",
            alpha=0.9,
        )

    residue_coords = np.array([point_by_index[idx] for idx in sorted(residue_indices)])
    phosphate_coords = np.array([point_by_index[idx] for idx in sorted(phosphate_indices)])
    donor_coords = np.array([point_by_index[idx] for idx in sorted(donor_indices)])
    acceptor_coords = np.array([point_by_index[idx] for idx in sorted(acceptor_indices)])

    if len(residue_coords):
        ax.scatter(residue_coords[:, 0], residue_coords[:, 1], s=110, color="#fcd34d", edgecolors="black", linewidths=0.4, zorder=2)
    if len(phosphate_coords):
        ax.scatter(phosphate_coords[:, 0], phosphate_coords[:, 1], s=170, color="#ef4444", edgecolors="black", linewidths=0.6, zorder=3)
    if len(donor_coords):
        ax.scatter(donor_coords[:, 0], donor_coords[:, 1], s=120, color="#2563eb", edgecolors="black", linewidths=0.4, zorder=4)
    if len(acceptor_coords):
        ax.scatter(acceptor_coords[:, 0], acceptor_coords[:, 1], s=120, color="#f59e0b", edgecolors="black", linewidths=0.4, zorder=4)

    label_indices = sorted(phosphate_indices | donor_indices | acceptor_indices)
    for atom_idx in label_indices:
        x, y = point_by_index[atom_idx]
        ax.text(
            x + 0.15,
            y + 0.15,
            f"{atom_array.res_name[atom_idx]}:{atom_array.atom_name[atom_idx]}",
            fontsize=8,
            zorder=5,
        )

    ax.set_title(f"Phosphosite H-bond sketch around {chain_id}{residue_id}")
    ax.set_aspect("equal")
    ax.set_axis_off()
    return ax


def make_sasa_view(atom_array, chain_id, residue_id, phosphate_atom_names, width=700, height=500):
    viewer = py3Dmol.view(width=width, height=height)
    viewer.addModel(
        to_cif_string(
            atom_array,
            include_entity_poly=False,
            _allow_ambiguous_bond_annotations=True,
        ),
        "mmcif",
    )
    viewer.setStyle(
        {},
        {"cartoon": {"color": "#94a3b8", "opacity": 0.3}, "stick": {"colorscheme": "lightgrayCarbon", "radius": 0.08}},
    )
    viewer.addSurface(
        py3Dmol.VDW,
        {"opacity": 0.35, "color": "white"},
        {"chain": chain_id, "resi": int(residue_id)},
    )
    viewer.setStyle(
        {"chain": chain_id, "resi": int(residue_id)},
        {"stick": {"colorscheme": "orangeCarbon", "radius": 0.18}},
    )

    phosphate_mask = (
        (atom_array.chain_id == chain_id)
        & (atom_array.res_id == residue_id)
        & np.isin(atom_array.atom_name, phosphate_atom_names)
    )
    for coord in atom_array.coord[phosphate_mask]:
        viewer.addSphere(
            {
                "center": {"x": float(coord[0]), "y": float(coord[1]), "z": float(coord[2])},
                "radius": 0.5,
                "color": "#ef4444",
                "opacity": 0.85,
            }
        )

    viewer.zoomTo({"chain": chain_id, "resi": int(residue_id)})
    return viewer


def compute_selection_sasa_metrics(atom_array, mask):
    if not np.any(mask):
        raise ValueError("Selection matched no atoms for SASA calculation")

    sasa_kwargs = {
        "probe_radius": SASA_PROBE_RADIUS,
        "vdw_radii": SASA_VDW_RADII,
        "point_number": SASA_POINT_NUMBER,
        "point_distr": SASA_POINT_DISTR,
    }
    full_complex_sasa = struc.sasa(atom_array, **sasa_kwargs)
    isolated_subset = atom_array[mask]
    isolated_sasa = struc.sasa(isolated_subset, **sasa_kwargs)

    total_iso = float(np.nansum(isolated_sasa))
    total_complex = float(np.nansum(full_complex_sasa[mask]))
    buried = float(total_iso - total_complex)
    fraction_buried = float(buried / total_iso) if total_iso > 0 else float("nan")
    return {
        "total_iso": total_iso,
        "total_complex": total_complex,
        "buried": buried,
        "fraction_buried": fraction_buried,
    }


def compute_selection_sasa_breakdown(atom_array, mask):
    if not np.any(mask):
        raise ValueError("Selection matched no atoms for SASA calculation")

    sasa_kwargs = {
        "probe_radius": SASA_PROBE_RADIUS,
        "vdw_radii": SASA_VDW_RADII,
        "point_number": SASA_POINT_NUMBER,
        "point_distr": SASA_POINT_DISTR,
    }
    full_complex_sasa = struc.sasa(atom_array, **sasa_kwargs)
    isolated_subset = atom_array[mask]
    isolated_sasa = struc.sasa(isolated_subset, **sasa_kwargs)

    rows = []
    selected_indices = np.flatnonzero(mask)
    for local_idx, atom_idx in enumerate(selected_indices):
        iso = float(isolated_sasa[local_idx])
        complex_val = float(full_complex_sasa[atom_idx])
        buried = float(iso - complex_val)
        rows.append(
            {
                "atom_label": atom_triplet_label(atom_array, atom_idx),
                "isolated_sasa": iso,
                "complex_sasa": complex_val,
                "buried_sasa": buried,
                "fraction_buried": float(buried / iso) if iso > 0 else float("nan"),
            }
        )
    return pd.DataFrame(rows)


In [ ]:
aligned_rf3_complex, binder_transform, binder_alignment_rmsd = align_mobile_on_binder_backbone(
    reference_complex,
    mobile_complex,
    binder_backbone_mask_ref,
    binder_backbone_mask_mobile,
)

binder_ref_idx, binder_mobile_idx = paired_common_indices(
    reference_complex,
    mobile_complex,
    binder_backbone_mask_ref,
    binder_backbone_mask_mobile,
)

rmsd_rows = [
    {
        "metric": "binder_backbone_alignment_rmsd",
        "rmsd_angstrom": binder_alignment_rmsd,
        "paired_atoms": len(binder_ref_idx),
    }
]

for metric_name, ref_mask, mobile_mask in [
    ("whole_peptide_ca_rmsd", peptide_ca_mask_ref, peptide_ca_mask_mobile),
    ("whole_peptide_all_atom_rmsd", peptide_mask_ref, peptide_mask_mobile),
    ("ptr_all_atom_rmsd", ptr_mask_ref, ptr_mask_mobile),
    ("po4_only_rmsd", po4_mask_ref, po4_mask_mobile),
]:
    metric_rmsd, paired_atoms = rmsd_for_masks(
        reference_complex,
        aligned_rf3_complex,
        ref_mask,
        mobile_mask,
        allow_mismatch=True,
    )
    rmsd_rows.append(
        {
            "metric": metric_name,
            "rmsd_angstrom": metric_rmsd,
            "paired_atoms": paired_atoms,
        }
    )

rmsd_metrics = pd.DataFrame(rmsd_rows)
display(rmsd_metrics)


A quick visual check helps here too. The first overlay shows the whole complex. The second zooms in on the phosphosite so the small shifts are easier to see by eye. Gray is the original RFD3 complex. Blue is the RF3 refold after aligning on the binder backbone.


In [ ]:
display(
    make_structure_overlay_view(
        reference_complex,
        aligned_rf3_complex,
        width=750,
        height=420,
    )
)

display(
    make_structure_overlay_view(
        reference_complex,
        aligned_rf3_complex,
        zoom_to_selection={"chain": TARGET_CHAIN_ID, "resi": int(ptm_residue_id)},
        width=750,
        height=420,
    )
)


In [ ]:
hbond_metrics = compute_phosphosite_hbond_metrics(
    aligned_rf3_complex,
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
    res_name="PTR",
)

hbond_summary = pd.DataFrame(
    [
        {
            "total_hbonds": hbond_metrics["total_hbonds"],
            "phosphosite_hbonds": hbond_metrics["phosphosite_hbonds"],
        }
    ]
)
display(hbond_summary)


Now zoom in on the actual phosphosite contacts. The plot below is a static H-bond sketch that always renders in the notebook, and the interactive 3D viewer is shown after it when the browser can load 3Dmol.js.


In [ ]:
hbond_detail_df = pd.DataFrame(hbond_metrics["phosphosite_records"])
if hbond_detail_df.empty:
    display(pd.DataFrame(columns=["donor_label", "hydrogen_label", "acceptor_label", "donor_acceptor_distance", "hydrogen_acceptor_distance"]))
else:
    display(
        hbond_detail_df[[
            "donor_label",
            "hydrogen_label",
            "acceptor_label",
            "donor_acceptor_distance",
            "hydrogen_acceptor_distance",
        ]].round(3)
    )

fig, ax = plt.subplots(figsize=(8, 7))
plot_hbond_contact_map(
    hbond_metrics["prepared_structure"],
    hbond_metrics["phosphosite_records"],
    chain_id=TARGET_CHAIN_ID,
    residue_id=ptm_residue_id,
    phosphate_atom_names=PHOSPHATE_ATOMS,
    ax=ax,
)
plt.tight_layout()
plt.show()

display(
    make_hbond_view(
        hbond_metrics["prepared_structure"],
        hbond_metrics["phosphosite_records"],
        chain_id=TARGET_CHAIN_ID,
        residue_id=ptm_residue_id,
        width=750,
        height=500,
    )
)


For SASA, the easiest way to read it is: how exposed is the group by itself, how exposed is it in the complex, and how much got buried when the complex formed. The summary table keeps both PTR and PO4. The atom-by-atom chart below focuses on PO4, since that is usually the business end of the phosphosite interaction.


In [ ]:
po4_sasa_metrics = compute_selection_sasa_metrics(aligned_rf3_complex, po4_mask_mobile)
ptr_sasa_metrics = compute_selection_sasa_metrics(aligned_rf3_complex, ptr_mask_mobile)

sasa_metrics = pd.DataFrame(
    [
        {"selection": "po4", **po4_sasa_metrics},
        {"selection": "ptr", **ptr_sasa_metrics},
    ]
)
display(sasa_metrics)


In [ ]:
po4_sasa_breakdown = compute_selection_sasa_breakdown(aligned_rf3_complex, po4_mask_mobile)

sasa_plot_df = pd.DataFrame(
    [
        {"selection": "PO4", **po4_sasa_metrics},
        {"selection": "PTR", **ptr_sasa_metrics},
    ]
)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sasa_plot_df.plot(
    x="selection",
    y=["total_iso", "total_complex", "buried"],
    kind="bar",
    ax=axes[0],
    color=["#cbd5e1", "#60a5fa", "#f59e0b"],
)
axes[0].set_ylabel("SASA")
axes[0].set_title("How much surface gets buried?")
axes[0].tick_params(axis="x", rotation=0)

po4_sasa_breakdown.plot(
    x="atom_label",
    y=["isolated_sasa", "complex_sasa"],
    kind="barh",
    ax=axes[1],
    color=["#cbd5e1", "#60a5fa"],
)
axes[1].set_xlabel("SASA")
axes[1].set_title("PO4 atom-by-atom exposure")

plt.tight_layout()
plt.show()

display(po4_sasa_breakdown.round(3))
display(
    make_sasa_view(
        aligned_rf3_complex,
        chain_id=TARGET_CHAIN_ID,
        residue_id=ptm_residue_id,
        phosphate_atom_names=PHOSPHATE_ATOMS,
        width=750,
        height=500,
    )
)


In [ ]:
rmsd_lookup = dict(zip(rmsd_metrics["metric"], rmsd_metrics["rmsd_angstrom"]))
po4_fraction_buried = po4_sasa_metrics["fraction_buried"]
phosphosite_hbonds = hbond_metrics["phosphosite_hbonds"]

metric_summary = pd.DataFrame(
    [
        {
            "peptide_ca_rmsd": rmsd_lookup["whole_peptide_ca_rmsd"],
            "peptide_ca_pass": rmsd_lookup["whole_peptide_ca_rmsd"] < FINAL_FILTER_MAX_PEPTIDE_CA_RMSD,
            "peptide_all_atom_rmsd": rmsd_lookup["whole_peptide_all_atom_rmsd"],
            "ptr_all_atom_rmsd": rmsd_lookup["ptr_all_atom_rmsd"],
            "po4_only_rmsd": rmsd_lookup["po4_only_rmsd"],
            "po4_fraction_buried": po4_fraction_buried,
            "po4_burial_pass": po4_fraction_buried > FINAL_FILTER_MIN_PO4_BURIAL,
            "phosphosite_hbonds": phosphosite_hbonds,
            "phosphosite_hbonds_pass": phosphosite_hbonds >= FINAL_FILTER_MIN_HBONDS,
            "overall_tutorial_pass": (
                rmsd_lookup["whole_peptide_ca_rmsd"] < FINAL_FILTER_MAX_PEPTIDE_CA_RMSD
                and po4_fraction_buried > FINAL_FILTER_MIN_PO4_BURIAL
                and phosphosite_hbonds >= FINAL_FILTER_MIN_HBONDS
            ),
        }
    ]
)
display(metric_summary)


## 8. Export the Key Files

Let's write out the files you'll probably want to keep from this run:
- the spoofed peptide target CIF
- the first RFD3 binder-target complex
- the selected LigandMPNN design
- the RF3-refolded complex


In [ ]:
spoofed_target_path = WORK_DIR / f"{EXAMPLE_NAME}_spoofed_target.cif"
rfd3_complex_path = WORK_DIR / f"{EXAMPLE_NAME}_rfd3_complex.cif"
rf3_complex_path = WORK_DIR / f"{EXAMPLE_NAME}_rf3_refolded.cif"
mpnn_base_path = WORK_DIR / f"{EXAMPLE_NAME}_mpnn_design_0"

to_cif_file(spoofed_target, spoofed_target_path)
to_cif_file(rfd3_complex, rfd3_complex_path)
selected_design.write_structure(base_path=mpnn_base_path)
to_cif_file(rf3_output.atom_array, rf3_complex_path)

print("Saved files:")
print(f" - {spoofed_target_path}")
print(f" - {rfd3_complex_path}")
print(f" - {mpnn_base_path}.cif")
print(f" - {rf3_complex_path}")


## What To Try Next

If you want to keep playing with this after the workshop, the easiest knobs to turn are:
1. Sample more RFD3 backbones and more LigandMPNN sequences.
2. Re-run the RF3 refold and compare the overlay, H-bonds, and SASA side by side.
3. Tighten the phosphosite interaction by checking whether the binder really reaches `O1P`, `O2P`, and `O3P`.
4. Save the candidates that still look good after those checks for deeper follow-up.
